In [1]:
%%writefile preprocessing.py
import re

def clean_text(txt):
    text = txt.replace("Subject:", "", 1).strip()
    text = re.sub(r'(?:_\s*){2,}', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_series(X):
    return X.apply(clean_text)

Writing preprocessing.py


In [2]:
import kagglehub
import os
import glob
import re
import numpy as np
import pandas as pd

from preprocessing import clean_text, clean_series
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings("ignore")

import seaborn as sns
import matplotlib.pyplot as plt

import joblib

In [3]:
path = kagglehub.dataset_download("jackksoncsie/spam-email-dataset")

csv_file = glob.glob(os.path.join(path, "*.csv"))
df = pd.read_csv(csv_file[0])
df.head()

100%|██████████| 2.86M/2.86M [00:00<00:00, 127MB/s]

Extracting files...


,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [4]:
def clean_text(txt):
    text = txt.replace("Subject:", "", 1).strip()
    text = re.sub(r'(?:_\s*){2,}', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
X_train, X_test, y_train, y_test = train_test_split(df.text, df.spam, test_size=0.2, random_state=42)

In [6]:
pipeline = Pipeline([
    ('clean', FunctionTransformer(clean_series)),
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english')),
    ('model', LogisticRegression())
])

In [7]:
pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)
y_pred = pipeline.predict(X_test)

In [8]:
print("Confusion Matrix\n")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

Confusion Matrix

[[854   2]
 [ 24 266]]

Classification Report

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       856
           1       0.99      0.92      0.95       290

    accuracy                           0.98      1146
   macro avg       0.98      0.96      0.97      1146
weighted avg       0.98      0.98      0.98      1146



In [9]:
param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__min_df': [1, 2],
    'model__C': [0.1, 1, 10]
}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)
y_pred=grid.predict(X_test)

In [10]:
print("Confusion Matrix\n")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report\n")
print(classification_report(y_test, y_pred))


Confusion Matrix

[[854   2]
 [  8 282]]

Classification Report

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       856
           1       0.99      0.97      0.98       290

    accuracy                           0.99      1146
   macro avg       0.99      0.99      0.99      1146
weighted avg       0.99      0.99      0.99      1146



In [11]:
joblib.dump(grid.best_estimator_, 'spam_pipeline.pkl')

['spam_pipeline.pkl']